# Train the AV contradiction detector on the AVID-Bench subset (Kaggle GPU)

One notebook for the whole run: clone the code, use the AVID-Bench subset you uploaded as a Kaggle
Dataset, extract frozen CLIP ViT-L/14 + CLAP features, train every model variant over 3 seeds, pick
the best model on **validation** macro-F1, and report it once on the held-out **test** split.

**Before you run**
1. Upload `data/avid/` from your laptop once as a **private Kaggle Dataset** (see
   `manifests/AVID_SUBSET.md`), then attach it here with *Add Input* (in the editor's right panel, under **Input**).
2. *Settings → Accelerator*: **GPU** (T4 or P100). *Settings → Internet*: **On** (pip, GitHub, Hugging Face weights).
3. `REPO_URL` below points at the project repo. For a **private** repo add a GitHub token as a Kaggle secret
   named `GITHUB_TOKEN` (*Add-ons → Secrets*); it is never printed or saved.
4. Use *Save Version → Save & Run All* so `/kaggle/working/output` is kept as the notebook output.

Labels follow AVID-Bench's `ground_truth.exists`: original audio → consistent, injected audio →
contradictory. The split (train/val/test) and every file come from `manifests/avid_subset.csv` in the repo.

In [ ]:
REPO_URL = "https://github.com/adhvaith267/Cross-Attention-Tensor-Fusion-for-Detecting-Audio-Visual-Semantic-Contradictions-in-Short-Form-Video.git"
BRANCH = "main"
SEEDS = "1 2 3"

OUT = "/kaggle/working/output"   # kept after the session: features, checkpoints, reports
CODE = "/tmp/av_contradiction"   # code lives outside /kaggle/working (not saved)

## 1. Get the code

In [ ]:
import os
import subprocess

url, token = REPO_URL, None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret attached: public repo
subprocess.run(["rm", "-rf", CODE], check=True)
r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE], capture_output=True, text=True)
log = r.stdout + r.stderr
print(log.replace(token, "***") if token else log)
assert r.returncode == 0, "Clone failed: check REPO_URL, BRANCH and the GITHUB_TOKEN secret"
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", REPO_URL], check=True)  # keep the token off disk
os.chdir(CODE)

## 2. Install and check the GPU

In [ ]:
!pip install -q -e ".[encoders]"
!python -m av_contradiction.cli.check_environment

## 3. Find the attached dataset

Looks for the folder under `/kaggle/input/*avid*` that contains `train_7239_segments/` and
`test_1171_segments/` (Kaggle may add a level when it unpacks a zip).

In [ ]:
## 3. Find the attached dataset

Searches everything under `/kaggle/input/` for the folder that contains `test_1171_segments/`
(Kaggle may add a level when it unpacks a zip). If nothing is found it lists what is attached.

import glob

MANIFEST = f"{CODE}/manifests/avid_subset.csv"
roots = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/test_1171_segments", recursive=True)})
if not roots:
    print("Nothing with test_1171_segments/ under /kaggle/input. What is attached:")
    for p in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30]:
        print("  ", p)
    raise SystemExit("Attach the AVID subset dataset: editor -> right panel -> Input -> + Add Input")
ROOT = roots[0]
n = len(glob.glob(f"{ROOT}/*_segments/*/*.mp4"))
print(f"dataset root: {ROOT} | {n} mp4 files")
assert n == 2600, f"expected 2600 videos, found {n}: is the upload complete?"

In [ ]:
os.makedirs(OUT, exist_ok=True)
!python -m av_contradiction.cli.audit_dataset --manifest {MANIFEST} --root-dir {ROOT} --out {OUT}/audit.json

## 5. Extract CLIP + CLAP features (downloads ~2.5 GB of weights once)

In [ ]:
COMMON = (f"--set experiment_name=avid --set data.manifest={MANIFEST} --set data.root_dir={ROOT} "
          f"--set data.feature_cache_dir={OUT}/cache --set training.checkpoint_dir={OUT}/checkpoints "
          f"--set evaluation.output_dir={OUT}/reports")
!python -m av_contradiction.cli.extract_features {COMMON} --set encoders.local_files_only=false

## 6. Train every variant and select the best model

Trains the proposed model (`xattn_tensor`), its ablations and the baselines on identical splits and
features, `SEEDS` times each. Every checkpoint is chosen on validation macro-F1; the overall best run
(highest **validation** macro-F1) is copied to `reports/avid/best_model/`. Test results never
influence any choice.

In [ ]:
!python -m av_contradiction.cli.run_ablations {COMMON} --seeds {SEEDS}

## 7. Results

In [ ]:
import json

from IPython.display import Image, Markdown, display

rep = f"{OUT}/reports/avid"
display(Markdown(open(f"{rep}/ablation_summary.md", encoding="utf-8").read()))
display(Image(f"{rep}/ablation_macro_f1.png"))
print(json.dumps(json.load(open(f"{rep}/best_model/best_model.json")), indent=2))

## 8. Download the best model

Download `best_model.zip` from the notebook's *Output* tab and unzip it into `models/<name>/` in the
project (for example `models/avid/`). The dashboard picks it up automatically. Commit that folder to
ship the trained model with the repository (`best.pt` is ~13 MB, without optimizer state).

In [ ]:
import shutil

print(shutil.make_archive(f"{OUT}/best_model", "zip", f"{OUT}/reports/avid/best_model"))